# Dataset Load

In [ ]:
from sklearn.datasets import load_wine
dataset = load_wine()

In [ ]:
print(dataset.DESCR)

.. _wine_dataset:

Wine recognition dataset
------------------------

**Data Set Characteristics:**

:Number of Instances: 178
:Number of Attributes: 13 numeric, predictive attributes and the class
:Attribute Information:
    - Alcohol
    - Malic acid
    - Ash
    - Alcalinity of ash
    - Magnesium
    - Total phenols
    - Flavanoids
    - Nonflavanoid phenols
    - Proanthocyanins
    - Color intensity
    - Hue
    - OD280/OD315 of diluted wines
    - Proline
    - class:
        - class_0
        - class_1
        - class_2

:Summary Statistics:

============================= ==== ===== ======= =====
                                Min   Max   Mean     SD
============================= ==== ===== ======= =====
Alcohol:                      11.0  14.8    13.0   0.8
Malic Acid:                   0.74  5.80    2.34  1.12
Ash:                          1.36  3.23    2.36  0.27
Alcalinity of Ash:            10.6  30.0    19.5   3.3
Magnesium:                    70.0 162.0    99.7  14.3

In [ ]:
x, y = dataset.data, dataset.target

# Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test =  train_test_split(x, y, test_size = 0.2, random_state = 2024)
x_train, x_val, y_train, y_val =  train_test_split(x_train, y_train, test_size = 0.2, random_state = 2024)

In [ ]:
print(x_train.shape, x_test.shape, x_val.shape)

(113, 13) (36, 13) (29, 13)


In [ ]:
print(y_train)
## 0,1,2의 multiclass label -> one hot encoding 필요

[0 0 2 1 2 2 0 2 0 0 0 1 1 0 1 2 0 0 0 0 0 1 0 2 1 2 1 0 1 0 2 2 1 1 1 1 0
 1 1 1 1 1 1 0 1 1 2 2 0 1 0 1 1 1 0 1 1 0 1 2 1 1 2 0 0 0 1 1 1 1 0 0 2 0
 2 1 1 2 1 0 0 0 1 0 1 2 1 0 0 1 2 1 2 2 0 0 2 2 1 2 0 1 1 1 0 1 1 2 1 0 1
 2 0]


# Multiclass Classification Modeling with PyTorch

## PyTorch Coding Style

1. Library Import
2. Dataset/Data Loader 생성/선언
3. Model Class 생성 / Model 선언
4. Optimizer / Loss선언
5. Model Training
6. Model Prediction
7. Model Evaluation

## 1.Library Import
- Onehot encoding 함수를 사용하기위해 functional 함수를 함께 import합니다

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, Dataset, DataLoader
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

## 2.Dataset / Data Loader 생성/선언

In [ ]:
x_train = torch.Tensor(x_train)
x_test = torch.Tensor(x_test)
x_val = torch.Tensor(x_val)

y_train = torch.from_numpy(y_train)
y_test = torch.from_numpy(y_test)
y_val = torch.from_numpy(y_val)

## one hot encoding
y_train = F.one_hot(y_train, num_classes = 3)
y_test = F.one_hot(y_test, num_classes = 3)
y_val = F.one_hot(y_val, num_classes = 3)


batch_size = 16
train_dataset = TensorDataset(x_train, y_train)
val_dataset = TensorDataset(x_val, y_val)
test_dataset = TensorDataset(x_test, y_test)


train_dataloader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
val_dataloader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False)
test_dataloader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False)

In [ ]:
y_train

tensor([[1, 0, 0],
        [1, 0, 0],
        [0, 0, 1],
        [0, 1, 0],
        [0, 0, 1],
        [0, 0, 1],
        [1, 0, 0],
        [0, 0, 1],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [0, 1, 0],
        [0, 1, 0],
        [1, 0, 0],
        [0, 1, 0],
        [0, 0, 1],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [0, 1, 0],
        [1, 0, 0],
        [0, 0, 1],
        [0, 1, 0],
        [0, 0, 1],
        [0, 1, 0],
        [1, 0, 0],
        [0, 1, 0],
        [1, 0, 0],
        [0, 0, 1],
        [0, 0, 1],
        [0, 1, 0],
        [0, 1, 0],
        [0, 1, 0],
        [0, 1, 0],
        [1, 0, 0],
        [0, 1, 0],
        [0, 1, 0],
        [0, 1, 0],
        [0, 1, 0],
        [0, 1, 0],
        [0, 1, 0],
        [1, 0, 0],
        [0, 1, 0],
        [0, 1, 0],
        [0, 0, 1],
        [0, 0, 1],
        [1, 0, 0],
        [0, 1, 0],
        [1, 0, 0],
        [0, 1, 0],
        [0, 

## 3.Model Class 생성 & Model 선언
- Functional Style

In [ ]:
class MulticlassModel(nn.Module):
    def __init__(self):
        super(MulticlassModel, self).__init__()

        self.linear_1 = nn.Linear(x_train.shape[1], 32)
        self.relu = nn.ReLU()
        self.linear_2 = nn.Linear(32,64)
        self.linear_3 = nn.Linear(64,128)
        self.linear_4 = nn.Linear(128,y_train.shape[1])
        self.softmax = nn.Softmax(dim = 1)

    def forward(self, x):
        x = self.linear_1(x)
        x = self.relu(x)
        x = self.linear_2(x)
        x = self.relu(x)
        x = self.linear_3(x)
        x = self.relu(x)
        x = self.linear_4(x)
        x = self.softmax(x)

        return x

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MulticlassModel().to(device)

## 4.Optimizer, Loss 선언
- Multiclass Classification Model은 loss function을 CrossEntropyLoss를 사용합니다.

In [ ]:
optimizer = optim.Adam(params = model.parameters(), lr = 0.001)

loss_fn = nn.CrossEntropyLoss().to(device)

## 5.Model Training

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
epochs = 100

for epoch in range(epochs):
  train_cost = 0
  train_accuracy = 0
  train_batch_length = len(train_dataloader)

  model.train()
  for train_inputs, train_labels in train_dataloader:

    ## train mode
    train_inputs = train_inputs.to(device)
    train_labels = train_labels.to(device)

    ## Model 예측
    train_preds = model(train_inputs)

    ## loss 계산
    train_loss = loss_fn(train_preds, train_labels.to(float))

    ## optimizer 초기화
    optimizer.zero_grad()

    ## Backpropagation
    train_loss.backward()

    ## weight를 next step으로 이동
    optimizer.step()

    train_cost += train_loss.item()
    train_preds = train_preds.argmax(axis = -1).detach().cpu().numpy()
    train_labels = train_labels.argmax(axis = -1).detach().cpu().numpy()

    train_accuracy += accuracy_score(train_labels, train_preds)

  train_cost /= train_batch_length
  train_accuracy /= train_batch_length

  ## validation set 평가

  val_cost = 0
  val_accuracy = 0
  val_batch_length = len(val_dataloader)

  model.eval() ## layer들을 inference 모드로 동작하도록 설정 변경
  with torch.no_grad(): ## optimizer가 gradient를 계산하는 연산을 수행하지 않고 inference목적으로만 사용하도록 하여 속도를 높임
    for val_inputs, val_labels in val_dataloader:

      val_inputs = val_inputs.to(device)
      val_labels = val_labels.to(device)

      val_preds = model(val_inputs)
      val_loss = loss_fn(val_preds, val_labels.to(float))
      val_cost += val_loss.item()


      val_preds = val_preds.argmax(axis = -1).detach().cpu().numpy()
      val_labels = val_labels.argmax(axis = -1).detach().cpu().numpy()
      val_accuracy += accuracy_score(val_labels, val_preds)

  val_cost /= val_batch_length
  val_accuracy /= val_batch_length

  print("{} / {} epochs => train loss : {:.4f}, train acc : {:.4f}, val loss : {:.4f}, val acc : {:.4f}".format(epoch+1, epochs, train_cost, train_accuracy, val_cost,  val_accuracy))

1 / 100 epochs => train loss : 1.2495, train acc : 0.2969, val loss : 1.2319, val acc : 0.3173
2 / 100 epochs => train loss : 1.2012, train acc : 0.3203, val loss : 1.0758, val acc : 0.3173
3 / 100 epochs => train loss : 0.9453, train acc : 0.5938, val loss : 1.0513, val acc : 0.3558
4 / 100 epochs => train loss : 0.9070, train acc : 0.6562, val loss : 1.0905, val acc : 0.4038
5 / 100 epochs => train loss : 0.9248, train acc : 0.6328, val loss : 0.9723, val acc : 0.5817
6 / 100 epochs => train loss : 0.9897, train acc : 0.5625, val loss : 0.9510, val acc : 0.6130
7 / 100 epochs => train loss : 0.8498, train acc : 0.7422, val loss : 0.9448, val acc : 0.6130
8 / 100 epochs => train loss : 0.8608, train acc : 0.7031, val loss : 0.9642, val acc : 0.5817
9 / 100 epochs => train loss : 0.8518, train acc : 0.7031, val loss : 0.9737, val acc : 0.5817
10 / 100 epochs => train loss : 0.9689, train acc : 0.5781, val loss : 0.9527, val acc : 0.6130
11 / 100 epochs => train loss : 0.9651, train acc

## 6.Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
y_test_pred = model(x_test.to(device))

In [ ]:
y_test_pred = y_test_pred.argmax(axis = -1).detach().cpu().numpy()

In [ ]:
y_test = y_test.argmax(axis = -1).detach().cpu().numpy()

In [ ]:
print(accuracy_score(y_test, y_test_pred))

0.3611111111111111


In [ ]:
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        11
           1       0.36      1.00      0.53        13
           2       0.00      0.00      0.00        12

    accuracy                           0.36        36
   macro avg       0.12      0.33      0.18        36
weighted avg       0.13      0.36      0.19        36



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


* 모델의 전반적인 정확도는 0.36 (36%)에 불과하다
* 구체적으로 Class 1의 Recall(재현율)은 1.00이다 (실제 Class 1 데이터 13개를 모델이 100%로 찾아냈다)
* 그러나, Class 0과 Class 2의 지표는 모두 0.00이다 (모델이 학습에 실패하고 하나의 클래스로만 예측값을 뱉어냈다)

